# Models Evaluation

## Traditional ML Models

In [3]:
from pathlib import Path

TRAIN_CSV = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/5000_Posts_Annotations - Combined_Dataset.csv")
TEST_CSV  = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/600_test_set.csv")

ID_COL    = "id"      
TITLE_COL = "title"
BODY_COL  = "body"


LABELS_COL = "Tags"

CANON_LABELS = [
    "Abuse", "Aggression", "Sexual", "Medical", "Mental Health",
    "Discrimination", "Pregnancy", "Not Applicable"
]

OUT_DIR = Path("/home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output dir:", OUT_DIR.resolve())

Output dir: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference


In [9]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Using device:", DEVICE)
print("✅ Output dir:", OUT_DIR.resolve())

✅ Using device: cpu
✅ Output dir: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference


In [4]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train:", train_df.shape, "Test:", test_df.shape)
display(train_df[[ID_COL, TITLE_COL, BODY_COL, LABELS_COL]].head(2))
display(test_df[[ID_COL, TITLE_COL, BODY_COL]].head(2))

Train: (4994, 6) Test: (600, 16)


,id,title,body,Tags
0,1ljxynj,Complications after abortion?,"Hi everyone, Ive read that abortions don’t cau...","Medical, Pregnancy, Mental Health"
1,1ljxtt8,Second MA abortion today and I'm absolutely te...,I'm having my second MA abortion today and I'm...,"Medical, Pregnancy, Mental Health"


,id,title,body
0,kg3jun,my assault ruins all of my relationships.,this is my first reddit post and i'm bad at ex...
1,77d66o,Me Too,After seeing all this hype over the #metoo thi...


In [5]:
def make_text(df):
    return (df[TITLE_COL].fillna("").astype(str) + " " + df[BODY_COL].fillna("").astype(str)).str.strip()

train_df["text"] = make_text(train_df)
test_df["text"]  = make_text(test_df)

print("text column created")

text column created


In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

def parse_labels(x):
    return [t.strip() for t in str(x).split(",") if t.strip()]

train_df["labels_list"] = train_df[LABELS_COL].fillna("").apply(parse_labels)

mlb = MultiLabelBinarizer(classes=CANON_LABELS)
Y_train = mlb.fit_transform(train_df["labels_list"])

X_train = train_df["text"].astype(str).values
X_test  = test_df["text"].astype(str).values

print("Y_train:", Y_train.shape)
print("Labels:", list(mlb.classes_))

Y_train: (4994, 8)
Labels: ['Abuse', 'Aggression', 'Sexual', 'Medical', 'Mental Health', 'Discrimination', 'Pregnancy', 'Not Applicable']


In [7]:
from sklearn.model_selection import train_test_split

X_tr, X_val, Y_tr, Y_val = train_test_split(
    X_train, Y_train,
    test_size=0.2,
    random_state=42
)
print("Train split:", X_tr.shape, "Val split:", X_val.shape)

Train split: (3995,) Val split: (999,)


### TF-IDF + LR

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

lr_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
        strip_accents="unicode"
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            solver="saga",
            max_iter=3000,
            class_weight="balanced",
            n_jobs=-1
        )
    ))
])

lr_pipe.fit(X_tr, Y_tr)
print("Logistic Regression model trained")

/home/ubuntu/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/ubuntu/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression model trained


In [20]:
import numpy as np
from sklearn.metrics import f1_score

val_proba = lr_pipe.predict_proba(X_val)

thresholds = np.arange(0.20, 0.81, 0.05)
best_thr, best_micro = None, -1

for thr in thresholds:
    Yv_pred = (val_proba >= thr).astype(int)
    micro = f1_score(Y_val, Yv_pred, average="micro", zero_division=0)
    if micro > best_micro:
        best_micro = micro
        best_thr = thr

print("Best LR threshold:", best_thr, "Val micro-F1:", round(best_micro, 4))

Best LR threshold: 0.49999999999999994 Val micro-F1: 0.7757


In [21]:
import pandas as pd

test_proba_lr = lr_pipe.predict_proba(X_test)
Y_pred_lr = (test_proba_lr >= best_thr).astype(int)

pred_labels_lr = mlb.inverse_transform(Y_pred_lr)

pred_lr_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].values,
    "pred_labels": [",".join(labels) if labels else "" for labels in pred_labels_lr],
    "threshold": best_thr
})

proba_cols = [f"proba_{lab}" for lab in mlb.classes_]
pred_lr_proba = pd.DataFrame(test_proba_lr, columns=proba_cols)

pred_lr_out = pd.concat([pred_lr_df, pred_lr_proba], axis=1)
LR_PATH = OUT_DIR / "predictions_tfidf_lr_test600.csv"
pred_lr_out.to_csv(LR_PATH, index=False)

print("Saved LR predictions:", LR_PATH.resolve())
display(pred_lr_out.head(3))

Saved LR predictions: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/predictions_tfidf_lr_test600.csv


,id,pred_labels,threshold,proba_Abuse,proba_Aggression,proba_Sexual,proba_Medical,proba_Mental Health,proba_Discrimination,proba_Pregnancy,proba_Not Applicable
0,kg3jun,"Abuse,Aggression,Sexual,Mental Health",0.5,0.862308,0.785555,0.742427,0.085497,0.749351,0.431987,0.105011,0.441694
1,77d66o,"Abuse,Aggression,Sexual,Mental Health",0.5,0.886917,0.736072,0.885444,0.163528,0.788677,0.350614,0.139335,0.254028
2,1lmsep9,Not Applicable,0.5,0.396362,0.465700,0.310570,0.233821,0.482805,0.265863,0.348081,0.579433


### TF-IDF + SVM

In [22]:
from sklearn.svm import LinearSVC

svm_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
        strip_accents="unicode"
    )),
    ("clf", OneVsRestClassifier(
        LinearSVC(class_weight="balanced")
    ))
])

svm_pipe.fit(X_train, Y_train)
print("Linear SVM trained")

Linear SVM trained


In [23]:
for i, label in enumerate(mlb.classes_):
    count = Y_train[:, i].sum()
    if count == 0:
        print(f"⚠️ Label '{label}' has ZERO training examples")

In [24]:
svm_scores = svm_pipe.decision_function(X_test)
Y_pred_svm = (svm_scores >= 0).astype(int)

pred_labels_svm = mlb.inverse_transform(Y_pred_svm)

pred_svm_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].values,
    "pred_labels": [",".join(labels) if labels else "" for labels in pred_labels_svm],
    "threshold": 0.0
})

# Save raw decision scores per label too
score_cols = [f"score_{lab}" for lab in mlb.classes_]
pred_svm_scores = pd.DataFrame(svm_scores, columns=score_cols)

pred_svm_out = pd.concat([pred_svm_df, pred_svm_scores], axis=1)
SVM_PATH = OUT_DIR / "predictions_tfidf_svm_test600.csv"
pred_svm_out.to_csv(SVM_PATH, index=False)

print("Saved SVM predictions:", SVM_PATH.resolve())
display(pred_svm_out.head(3))

Saved SVM predictions: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/predictions_tfidf_svm_test600.csv


,id,pred_labels,threshold,score_Abuse,score_Aggression,score_Sexual,score_Medical,score_Mental Health,score_Discrimination,score_Pregnancy,score_Not Applicable
0,kg3jun,"Abuse,Sexual,Mental Health",0.0,0.741901,-0.836789,0.264626,-1.037100,0.641000,-0.375343,-1.122656,-0.335727
1,77d66o,"Abuse,Sexual,Mental Health",0.0,1.032490,-0.764224,0.925201,-0.607432,0.941310,-0.621776,-0.595512,-0.928600
2,1lmsep9,Not Applicable,0.0,-0.533150,-1.226374,-0.633979,-0.838246,-0.245532,-0.776147,-0.510205,0.325181


### Saving and Sanity Checks

In [25]:
import joblib, json

joblib.dump(lr_pipe, OUT_DIR / "tfidf_lr.joblib")
joblib.dump(svm_pipe, OUT_DIR / "tfidf_svm.joblib")
joblib.dump(mlb, OUT_DIR / "mlb.joblib")

meta = {
    "labels": list(mlb.classes_),
    "lr_best_threshold": float(best_thr),
    "tfidf": {"ngram_range": [1,2], "min_df": 2, "max_df": 0.9, "sublinear_tf": True}
}
with open(OUT_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved model artifacts to:", OUT_DIR.resolve())

Saved model artifacts to: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference


In [26]:
import numpy as np

lr_counts  = np.array([len(x) for x in pred_labels_lr])
svm_counts = np.array([len(x) for x in pred_labels_svm])

print("LR avg labels/post:", lr_counts.mean().round(2), "min:", lr_counts.min(), "max:", lr_counts.max())
print("SVM avg labels/post:", svm_counts.mean().round(2), "min:", svm_counts.min(), "max:", svm_counts.max())

print("\nLR label-count distribution:")
print(pd.Series(lr_counts).value_counts().sort_index())

print("\nSVM label-count distribution:")
print(pd.Series(svm_counts).value_counts().sort_index())

LR avg labels/post: 2.99 min: 0 max: 6
SVM avg labels/post: 2.42 min: 0 max: 6

LR label-count distribution:
0      4
1     38
2    131
3    238
4    165
5     23
6      1
Name: count, dtype: int64

SVM label-count distribution:
0      7
1     83
2    180
3    313
4     15
5      1
6      1
Name: count, dtype: int64


## Transformers

In [26]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import json

In [32]:
import numpy as np 

def build_ds(texts, labels=None):
    d = {"text": list(texts)}
    if labels is not None:
        d["labels"] = labels.astype(np.float32).tolist()
    return Dataset.from_dict(d)

def sigmoid(x):
    return 1/(1+np.exp(-x))

def tune_threshold(val_logits, y_true):
    probs = sigmoid(val_logits)
    best_thr, best_micro = 0.5, -1
    for thr in np.arange(0.20, 0.81, 0.05):
        y_pred = (probs >= thr).astype(int)
        micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
        if micro > best_micro:
            best_micro, best_thr = micro, thr
    return float(best_thr), float(best_micro)

def save_predictions(model_key, threshold, probs, pred_labels):
    model_dir = OUT_DIR / model_key
    model_dir.mkdir(parents=True, exist_ok=True)

    out_df = pd.DataFrame({
        ID_COL: test_df[ID_COL].values,
        "model": model_key,
        "threshold": threshold,
        "pred_labels": [",".join(x) if x else "" for x in pred_labels],
    })
    proba_cols = [f"proba_{lab}" for lab in CANON_LABELS]
    out_probs = pd.DataFrame(probs, columns=proba_cols)

    out_full = pd.concat([out_df, out_probs], axis=1)
    path = model_dir / "predictions_test600.csv"
    out_full.to_csv(path, index=False)
    print("✅ Saved:", path.resolve())
    return path, model_dir

In [28]:
def tokenize_dataset(ds, tokenizer, max_length=256, has_labels=True):
    def tok(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

    ds2 = ds.map(tok, batched=True)

    if has_labels:
        # ✅ Force labels to float32 for BCEWithLogitsLoss (multi-label)
        ds2 = ds2.map(lambda b: {"labels": [list(map(float, x)) for x in b["labels"]]}, batched=True)

    cols = ["input_ids", "attention_mask"] + (["labels"] if has_labels else [])
    ds2.set_format(type="torch", columns=cols)
    return ds2

In [29]:
from transformers import TrainingArguments
import inspect

def make_training_args(output_dir, **kwargs):
    """
    Create TrainingArguments in a version-safe way.
    Some versions use evaluation_strategy; others use eval_strategy.
    """
    sig = inspect.signature(TrainingArguments.__init__).parameters

    # rename if needed
    if "evaluation_strategy" in kwargs and "evaluation_strategy" not in sig and "eval_strategy" in sig:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")

    # remove unsupported keys (safety)
    safe_kwargs = {k: v for k, v in kwargs.items() if k in sig}
    dropped = set(kwargs) - set(safe_kwargs)
    if dropped:
        print("⚠️ Dropped unsupported TrainingArguments keys:", dropped)

    return TrainingArguments(output_dir=output_dir, **safe_kwargs)

In [30]:
batch = next(iter(trainer_bert.get_train_dataloader()))
print(batch["labels"].dtype, batch["labels"].shape)

torch.int64 torch.Size([8, 8])


### BERT

In [33]:
MODEL_NAME_BERT = "bert-base-uncased"
MODEL_KEY_BERT  = "bert-base-uncased"

tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME_BERT, use_fast=True)

ds_tr_bert  = tokenize_dataset(build_ds(X_tr,  Y_tr), tokenizer_bert, max_length=256, has_labels=True)
ds_val_bert = tokenize_dataset(build_ds(X_val, Y_val), tokenizer_bert, max_length=256, has_labels=True)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_BERT,
    num_labels=len(CANON_LABELS),
    problem_type="multi_label_classification"
)

args_bert = make_training_args(
    output_dir=str(OUT_DIR / MODEL_KEY_BERT / "ckpt"),
    evaluation_strategy="epoch",   # will auto-map if needed
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42
)

trainer_bert = Trainer(
    model=bert_model,
    args=args_bert,
    train_dataset=ds_tr_bert,
    eval_dataset=ds_val_bert
)

trainer_bert.train()

val_logits_bert = trainer_bert.predict(ds_val_bert).predictions
best_thr_bert, best_micro_bert = tune_threshold(val_logits_bert, Y_val)
print(f"✅ BERT best_thr={best_thr_bert} val_microF1={best_micro_bert:.4f}")

Map:   0%|          | 0/3995 [00:00<?, ? examples/s]

Map:   0%|          | 0/3995 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/ubuntu/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
X_test = test_df["text"].astype(str).values
ds_test_bert = tokenize_dataset(build_ds(X_test), tokenizer_bert, max_length=256, has_labels=False)

test_logits_bert = trainer_bert.predict(ds_test_bert).predictions
test_probs_bert  = sigmoid(test_logits_bert)
Y_pred_bert      = (test_probs_bert >= best_thr_bert).astype(int)
pred_labels_bert  = mlb.inverse_transform(Y_pred_bert)

pred_path_bert, model_dir_bert = save_predictions(MODEL_KEY_BERT, best_thr_bert, test_probs_bert, pred_labels_bert)

trainer_bert.save_model(str(model_dir_bert / "final_model"))
tokenizer_bert.save_pretrained(str(model_dir_bert / "final_model"))
(model_dir_bert / "meta.json").write_text(json.dumps({
    "model": MODEL_NAME_BERT, "best_thr": best_thr_bert, "val_microF1": best_micro_bert, "labels": CANON_LABELS
}, indent=2))

### RoBERTa

## LLMS

### GPT